In [0]:
# Add widgets (parameters)
from pyspark.sql import functions as F

dbutils.widgets.text("silver_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/silver/events")
dbutils.widgets.text("gold_product_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/gold/product_perf")
dbutils.widgets.text("gold_category_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/gold/category_perf")

silver_path = dbutils.widgets.get("silver_path")
gold_product_path = dbutils.widgets.get("gold_product_path")
gold_category_path = dbutils.widgets.get("gold_category_path")


In [0]:
silver_df = spark.read.format("delta").load(silver_path)

product_perf = (
    silver_df
    .groupBy("product_id", "brand", "category_code")
    .agg(
        F.countDistinct(F.when(F.col("event_type") == "view", F.col("user_id"))).alias("views"),
        F.countDistinct(F.when(F.col("event_type") == "purchase", F.col("user_id"))).alias("purchases"),
        F.sum(F.when(F.col("event_type") == "purchase", F.col("price"))).alias("revenue")
    )
    .withColumn("views", F.coalesce(F.col("views"), F.lit(0)))
    .withColumn("purchases", F.coalesce(F.col("purchases"), F.lit(0)))
    .withColumn("revenue", F.coalesce(F.col("revenue"), F.lit(0.0)))
    .withColumn(
        "conversion_rate",
        F.when(F.col("views") == 0, F.lit(0.0)).otherwise(F.col("purchases") / F.col("views") * 100)
    )
)

category_perf = (
    silver_df
    .groupBy("category_code")
    .agg(
        F.count("*").alias("events"),
        F.sum(F.when(F.col("event_type") == "purchase", F.col("price"))).alias("revenue"),
        F.countDistinct("user_id").alias("active_users")
    )
    .withColumn("revenue", F.coalesce(F.col("revenue"), F.lit(0.0)))
)

product_perf.write.format("delta").mode("overwrite").save(gold_product_path)
category_perf.write.format("delta").mode("overwrite").save(gold_category_path)

dbutils.notebook.exit(
    f"OK gold product_rows={product_perf.count()} category_rows={category_perf.count()}"
)
